In [2]:
pip install -U jax

   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.9 MB 4.6 MB/s eta 0:00:01
   ---------- ----------------------------- 0.8/2.9 MB 4.7 MB/s eta 0:00:01
   ---------- ----------------------------- 0.8/2.9 MB 4.7 MB/s eta 0:00:01
   --------------------- ------------------ 1.6/2.9 MB 1.8 MB/s eta 0:00:01
   ------------------------- -------------- 1.8/2.9 MB 1.8 MB/s eta 0:00:01
   -------------------------------- ------- 2.4/2.9 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 2.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/59.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/59.8 MB ? eta -:--:--
    --------------------------------------- 1.0/59.8 MB 2.7 MB/s eta 0:00:23
   - -------------------------------------- 1.6/59.8 MB 2.8 MB/s eta 0:00:21
   - -------------------------------------- 2.1/59.8 MB 2.8 MB/s eta 0:00:21
   - -----------------------

Hybrid Risk Model and Volatility
This risk model is a sophisticated two-part system designed to accurately calculate the intrinsic risk of an aircraft part and use that risk to improve price predictions.

Component	Purpose	Key Factor Inputs
1. Conceptual Risk Model (JAX)	To quantify the component's internal (operational) and external (market) liability into a single, objective score.	Time Since New (TSN), Cycle Since Repair (CSR), Time Life (TL), Market Volatility (APVI), Traceability.
2. XGBoost Predictor	To combine the calculated risk score with other features to produce the final, market-adjusted sale price.

**Create model to calculate market volatility**

In [3]:
pip install yfinance


  Using cached yfinance-0.2.66-py2.py3-none-any.whl.metadata (6.0 kB)
  Using cached multitasking-0.0.12-py3-none-any.whl
     ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
     ------ --------------------------------- 0.5/3.0 MB 5.4 MB/s eta 0:00:01
     ------ --------------------------------- 0.5/3.0 MB 5.4 MB/s eta 0:00:01
     ---------- ----------------------------- 0.8/3.0 MB 1.0 MB/s eta 0:00:03
     ------------- -------------------------- 1.0/3.0 MB 1.4 MB/s eta 0:00:02
     ------------- -------------------------- 1.0/3.0 MB 1.4 MB/s eta 0:00:02
     ----------------- ---------------------- 1.3/3.0 MB 938.9 kB/s eta 0:00:02
     --------------------------- ------------ 2.1/3.0 MB 1.4 MB/s eta 0:00:01
     -------------------------------------- - 2.9/3.0 MB 1.7 MB/s eta 0:00:01
     ---------------------------------------- 3.0/3.0 MB 1.7 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
 

In [4]:
pip install pandas_datareader


  Using cached pandas_datareader-0.10.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached lxml-6.0.2-cp313-cp313-win_amd64.whl.metadata (3.7 kB)
Using cached pandas_datareader-0.10.0-py3-none-any.whl (109 kB)
Using cached lxml-6.0.2-cp313-cp313-win_amd64.whl (4.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install arch

   ---------------------------------------- 0.0/929.7 kB ? eta -:--:--
   --------------------------------- ------ 786.4/929.7 kB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 929.7/929.7 kB 5.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.5 MB 7.6 MB/s eta 0:00:02
   ------------ --------------------------- 2.9/9.5 MB 7.4 MB/s eta 0:00:01
   ------------- -------------------------- 3.1/9.5 MB 7.4 MB/s eta 0:00:01
   -------------- ------------------------- 3.4/9.5 MB 4.1 MB/s eta 0:00:02
   ------------------ --------------------- 4.5/9.5 MB 4.3 MB/s eta 0:00:02
   -------------------------- ------------- 6.3/9.5 MB 4.9 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.5 MB 4.9 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.5 MB 4.8 MB/s eta 0:00:01
   ------------------------------------ --- 8.7/9.5 MB 4.5 MB/s eta 0:00:01
   -------------

This algorithm calculates the Aircraft Parts Volatility Index (APVI), a composite measure of market and industry volatility for aircraft components. First, it downloads historical stock prices for key suppliers and an industry ETF, then computes 90-day rolling realised volatility for each, capturing recent market fluctuations. It also fetches the Producer Price Index (PPI) for aircraft parts from FRED and calculates its annualised volatility, reflecting supply-side cost variability. To account for short-term conditional volatility in the market, it applies a GARCH(1,1) model to the ETF returns, producing a forecasted daily volatility. The APVI itself is computed as the simple average of the mean equity volatility and the PPI volatility, providing a single, interpretable metric. Finally, the algorithm produces a publication-ready figure showing market volatility, PPI volatility, and the APVI, facilitating visual comparison between market risk, supply-side volatility, and the combined sector volatility index over time.

Sources for Volatility Calculation
Here is the complete list of companies and the Exchange Traded Fund (ETF) corresponding to the tickers used in the volatility calculation:

Ticker	Full Company / ETF Name	Brief Description
TDG	TransDigm Group Incorporated	A major global designer, producer, and supplier of highly engineered aircraft components for commercial and military applications.
SPR	Spirit AeroSystems Holdings, Inc.	One of the world's largest independent non-OEM (Original Equipment Manufacturer) suppliers of commercial aerostructures.
HXL	Hexcel Corporation	A leading global manufacturer of advanced composite materials, crucial for creating lightweight and strong aerospace components.
HON	Honeywell International Inc.	A large, diversified technology and manufacturing company with a significant Aerospace division providing engines, avionics, and services.
XAR	SPDR S&P Aerospace & Defense ETF	An Exchange Traded Fund that tracks a broad index of companies within the U.S. aerospace and defense industry, used as a sector benchmark.

A **ticker symbol** (or just "ticker") is an **abbreviation** used to uniquely identify publicly traded securities—most commonly stocks, bonds, and ETFs—on a particular stock exchange.

It's essentially a short, recogniable code, usually consisting of **two to five letters**, that allows traders and investors to quickly reference a company or financial instrument.

### Key Characteristics:

* **Uniqueness:** Every publicly traded company or financial product has a unique ticker for each exchange it is listed on. For example, Apple Inc. trades as **AAPL** on NASDAQ.
* **Identification:** Tickers are used for trading, looking up prices, and viewing historical data.
* **Mnemonic Value:** While some tickers are simple abbreviations (like **HON** for Honeywell), others are designed to be memorable or descriptive (like **GOOG** for Google/Alphabet).
* **Exchange Specificity:** Tickers can sometimes be re-used across different exchanges, but the combination of the ticker plus the exchange is always unique.

In your volatility calculation, the tickers used are:
* **TDG, SPR, HXL, HON:** These are the tickers for individual aerospace companies.
* **XAR:** This is the ticker for the SPDR S\&P Aerospace \& Defense **ETF** (Exchange Traded Fund).

In [6]:
import pandas as pd
import numpy as np
import yfinance as yf
import jax.numpy as jnp
from jax import jit, lax
from jax.scipy.optimize import minimize
import plotly.graph_objects as go

tickers = ['TDG', 'SPR', 'HXL', 'HON', 'XAR']  # See above comments
start = "2015-01-01"
end = "2025-01-01"

price_list = []

for t in tickers:
    df = yf.download(t, start=start, end=end, progress=False)
    if 'Adj Close' in df.columns:
        series = df['Adj Close'].copy()
    else:
        series = df.iloc[:, 0].copy()
    series.name = t
    price_list.append(series)

adj_close = pd.concat(price_list, axis=1)
returns = np.log(adj_close / adj_close.shift(1)).dropna()

returns_jax = jnp.array(returns.values)
rolling_window = 90
sqrt252 = jnp.sqrt(252)

def rolling_std_jax(arr, window=90):
    n_days, n_tickers = arr.shape
    vols = []
    for i in range(n_days - window + 1):
        slice_ = arr[i:i+window, :]
        vol = jnp.std(slice_, axis=0) * sqrt252
        vols.append(vol)
    return jnp.array(vols)

realized_vol_jax = rolling_std_jax(returns_jax, window=rolling_window)

realized_vol_df = pd.DataFrame(
    realized_vol_jax,
    columns=[t + "_vol" for t in tickers],
    index=returns.index[rolling_window-1:]
)

# Fetch PPI for Aircraft Parts 
ppi_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=PCU336413336413P"
ppi = pd.read_csv(ppi_url)
ppi.rename(columns={ppi.columns[0]: 'DATE', ppi.columns[1]: 'PPI'}, inplace=True)
ppi['DATE'] = pd.to_datetime(ppi['DATE'])
ppi.set_index('DATE', inplace=True)
ppi['ppi_returns'] = np.log(ppi['PPI'] / ppi['PPI'].shift(1))
ppi['ppi_vol'] = ppi['ppi_returns'].rolling(12).std() * np.sqrt(12)

''' The following function This function implements a 
GARCH (Generalized Autoregressive Conditional Heteroskedasticity) 
model for forecasting volatility in financial time series data. 
Here's what it does:
Purpose: Predicts the conditional variance (volatility) of returns over time, 
which is crucial for risk management and pricing in finance.'''

@jit
def garch_forecast(params, returns):
    omega, alpha, beta = jnp.exp(params)
    T = len(returns)
    h = jnp.zeros(T)
    h = h.at[0].set(jnp.var(returns))
    
    def step(t, h):
        h = h.at[t].set(omega + alpha * returns[t-1]**2 + beta * h[t-1])
        return h

    h = lax.fori_loop(1, T, step, h)
    return h

@jit
def garch_neg_loglik(params, returns):
    h = garch_forecast(params, returns)
    ll = 0.5 * jnp.sum(jnp.log(h[1:]) + (returns[1:]**2)/h[1:])
    return ll

def garch_fit_jax(returns):
    init_params = jnp.log(jnp.array([0.01, 0.05, 0.9]))
    result = minimize(garch_neg_loglik, init_params, args=(returns,), method="BFGS")
    h = garch_forecast(result.x, returns)
    omega, alpha, beta = jnp.exp(result.x)
    return h, (omega, alpha, beta)

# XAR (daily) - # TODO this needs to be computationationaly solved first - JAX implt
xar_returns = jnp.array(returns['XAR'].dropna() * 100)  # percent
h_xar, params_xar = garch_fit_jax(xar_returns)
garch_vol = jnp.sqrt(h_xar[-1])

print(f"\nLatest JAX GARCH(1,1) forecasted daily volatility for XAR: {garch_vol:.3f}%")
print(f"GARCH parameters: omega={params_xar[0]:.4f}, alpha={params_xar[1]:.4f}, beta={params_xar[2]:.4f}")

#Calculate the APVI
latest_equity_vol = float(jnp.mean(realized_vol_jax[-1, :]))
latest_ppi_vol = float(ppi['ppi_vol'].iloc[-1])
if np.isnan(latest_ppi_vol):
    latest_ppi_vol = 0
apvi_jax = (latest_equity_vol + latest_ppi_vol) / 2
print(f"\nComposite Aircraft Parts Volatility Index (APVI): {apvi_jax:.3f}")

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=realized_vol_df.index,
    y=realized_vol_df['XAR_vol'],
    mode='lines',
    name='XAR 90d Realized Vol',
    line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=ppi.index,
    y=ppi['ppi_vol'],
    mode='lines',
    name='PPI Volatility (Aircraft Parts)',
    line=dict(color='red'),
    yaxis='y2'
))

fig.add_trace(go.Scatter(
    x=[realized_vol_df.index[0], realized_vol_df.index[-1]],
    y=[apvi_jax, apvi_jax],
    mode='lines',
    name='Composite APVI',
    line=dict(color='green', dash='dash')
))

fig.update_layout(
    title="Aircraft Parts Industry – Market vs PPI Volatility",
    xaxis_title="Date",
    yaxis_title="XAR Realized Vol",
    yaxis2=dict(title="PPI Volatility", overlaying='y', side='right'),
    legend=dict(x=0.01, y=0.99)
)

fig.show()


C:\Users\dean.foulds\AppData\Local\Temp\ipykernel_18176\560770336.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(t, start=start, end=end, progress=False)
C:\Users\dean.foulds\AppData\Local\Temp\ipykernel_18176\560770336.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(t, start=start, end=end, progress=False)
C:\Users\dean.foulds\AppData\Local\Temp\ipykernel_18176\560770336.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(t, start=start, end=end, progress=False)
C:\Users\dean.foulds\AppData\Local\Temp\ipykernel_18176\560770336.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(t, start=start, end=end, progress=False)
C:\Users\dean.foulds\AppData\Local\Temp\ipykernel_18176\560770336.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df 


Latest JAX GARCH(1,1) forecasted daily volatility for XAR: 1.528%
GARCH parameters: omega=0.0463, alpha=0.0870, beta=0.8864

Composite Aircraft Parts Volatility Index (APVI): 0.121


 The redline represents the volativity in aircraft parts sine approx. 1985.  Genererally world events causing supply chain issues. Data from FRED(Nov, 2025).  A Good example post 2021/2022 saw a sudden increase in demand as supply chains struggled to keep up following COVID. The blue line represents market equity volatility hence the 2020 spike due to COVID.  Other factors could include war in Ukraine or blocked canal, Ever Given ship.  The greeline just gives a composit baseline an aggregation of XAR and PPI should be used as an input into the risk algorithm. 

In [7]:
import jax.numpy as jnp
from jax import grad, jit, vmap



# We now define seven risk factors (X1 to X7):
# X1: Traceability Risk
# X2: Time Since New Risk (TSN)
# X3: Cycle Since New Risk (CSN)
# X4: Time Since Repair Risk (TSR)
# X5: Cycle Since Repair Risk (CSR)
# X6: Remaining Life Risk (based on Time Life/Max Life)
# X7: Market Volatility Risk

# Initial Weights (W): Must have 8 elements. Adjust these based on domain expertise.
# Prioritizing Remaining Life (X6), Traceability (X1), and CSN (X3) in this initial guess.
W = jnp.array([0.25, 0.10, 0.15, 0.05, 0.05, 0.30, 0.10]) # Sums to 1.0

# Threshold (T): Used to shift the result before the sigmoid - ensures a baseline high ish after 0.5
THRESHOLD = 0.6 

@jit
def normalize_factors(raw_input):
    """
    Converts raw part data into normalized risk factors (0-1).
    Input order (7 inputs):
    [0] Traceability (1-3)
    [1] Time Since New (TSN)
    [2] Cycle Since New (CSN)
    [3] Time Since Repair (TSR)
    [4] Cycle Since Repair (CSR)
    [5] Current Time Life (Hours/Cycles) 
    [6] Max Certified Time Life (Hours/Cycles) 
    [7] Market Volatility (0-1)
    # manufactured date
    NOTE: The number of inputs must match the expected array size. 
    The raw input array must now contain 7 values (7 risk metrics + 1 max life for X6 calculation).
    """
    
    # Check the size of the raw input array 
    # This will be removed if data not avalable or other labels can be found in Quantum
    if raw_input.shape[0] != 8:
        raise ValueError("Input array must contain 8 elements for all required factors.")

    raw_trace, tsn, csn, tsr, csr, current_life, max_life, apvi_jax = raw_input

    # --- X1: Traceability Risk (Inverse) ---
    # Risk = 1 - (Normalized Traceability Score)
    #Ensures the input score is constrained to the expected range of 1.0 to 3.0. This prevents erroneous or out-of-bounds
    # input data from breaking the normalization or creating unrealistic risk scores
    X1 = 1.0 - (jnp.clip(raw_trace, 1.0, 3.0) - 1.0) / 2.0

    # --- X2: Time Since New (TSN) Risk ---
    # Assuming max risk for time-based components is reached after 20,000 hours/units.
    MAX_TSN = 20000.0
    X2 = jnp.clip(tsn / MAX_TSN, 0.0, 1.0)

    # --- X3: Cycle Since New (CSN) Risk ---
    # Assuming max risk for cycle-based components is reached after 10,000 cycles.
    MAX_CSN = 10000.0
    X3 = jnp.clip(csn / MAX_CSN, 0.0, 1.0)
    
    # Risk as time since repair increases; assuming max risk after 5 years/units of time.
    MAX_TSR = 5.0
    X4 = jnp.clip(tsr / MAX_TSR, 0.0, 1.0)
    
    # Risk  as cycles since repair increase; assuming max risk after 1,000 cycles, part dependant
    MAX_CSR = 1000.0
    X5 = jnp.clip(csr / MAX_CSR, 0.0, 1.0)
    
    # Risk increases as Life Used increases.
    life_used_ratio = current_life / max_life
    X6 = jnp.clip(life_used_ratio, 0.0, 1.0) # Note: Life Used is the risk factor, not Remaining Life
    
    # Key point here is live data.
    MAX_VOLATILITY = 0.5 
    X7 = jnp.clip(apvi_jax / MAX_VOLATILITY, 0.0, 1.0)

    return jnp.array([X1, X2, X3, X4, X5, X6, X7])


#Risk Algorithm (Mathematical Model)

@jit
def risk_algorithm(W, THRESHOLD, normalized_factors):
    """
    Calculates the Total Risk Score (R) using the weighted sum and sigmoid if needed.
    """
    # Check that weights and factors match in size
    if W.shape[0] != normalized_factors.shape[0]:
        raise ValueError("Weights array and Normalized Factors array must have the same length (7).")
        
    # Weighted Sum of Factors (dot product of weights and factors)
    weighted_sum = jnp.dot(W, normalized_factors)
    
    # Sigmoid function: R = 1 / (1 + exp(-(z)))
    R = 1.0 / (1.0 + jnp.exp(-(weighted_sum - THRESHOLD)))
    
    return R



# Example Part E (Low Risk):
# [Traceability(3), TSN(1000), CSN(500), TSR(0.1), CSR(10), Current Life(5000), Max Life(20000), Volatility(0.05)]
part_data_E = jnp.array([3.0, 1000.0, 500.0, 0.1, 10.0, 5000.0, 20000.0, 0.05]) 

# Example Part F (High Risk):
# [Traceability(1), TSN(18000), CSN(8000), TSR(4.0), CSR(800), Current Life(18000), Max Life(20000), Volatility(0.4)]
part_data_F = jnp.array([1.0, 18000.0, 8000.0, 4.0, 800.0, 18000.0, 20000.0, 0.4]) 

normalized_E = normalize_factors(part_data_E)
risk_E = risk_algorithm(W, THRESHOLD, normalized_E)

normalized_F = normalize_factors(part_data_F)
risk_F = risk_algorithm(W, THRESHOLD, normalized_F)

print(f"--- Part E (Low Risk) ---")
print(f"Normalized Risk Factors (X1-X7): {normalized_E}")
print(f"Risk Score: {risk_E:.4f}")

print(f"\n--- Part F (High Risk) ---")
print(f"Normalized Risk Factors (X1-X7): {normalized_F}")
print(f"Risk Score: {risk_F:.4f}")

--- Part E (Low Risk) ---
Normalized Risk Factors (X1-X7): [0.   0.05 0.05 0.02 0.01 0.25 0.1 ]
Risk Score: 0.3773

--- Part F (High Risk) ---
Normalized Risk Factors (X1-X7): [1.         0.9        0.79999995 0.8        0.8        0.9
 0.8       ]
Risk Score: 0.5720
